In [ ]:
# pip install langchain langchain-community faiss-cpu sentence-transformers huggingface_hub pypdf2 ollama

In [5]:

from langchain_community.document_loaders import PyPDFLoader  # reads PDFs page by page
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
import os
import random 

# 1) Load PDF
pdf_path = r"C:\Users\Rohan\Desktop\Rohan\ML\RAG\mini-Animal-Kingdom.pdf"
assert os.path.exists(pdf_path), f"File not found: {pdf_path}"

def text_formatter(text: str) -> str:
    text = text.replace("\n", " ")  # replace new line characters with space
    text = " ".join(text.split())  # remove multiple spaces
    text = text.replace("⚪" , "")  # remove special characters
    # if text and text[0].isdigit():
    #     text = text.split(" ", 1)[1] # remove first page number if exists 
    return text

loader = PyPDFLoader(pdf_path)
docs = loader.load()  # one Document per page

for d in docs:
    d.page_content = text_formatter(d.page_content or "")

print("Pages:", len(docs))
print("Random page preview:\n",  random.choice(docs).page_content[:600])


#2) Split 
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = splitter.split_documents(docs)
print("Document splits:", len(splits))

# 3) Embeddings + Vector store
embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(splits, embedding=embeddings) # giving splits to create the vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

#4) Prompt
prompt = ChatPromptTemplate.from_template(
    """You are a helpful AI assistant.
Explain the answer as if teaching a young student, using simple words and short sentences.
Only use the context below. If the answer isn't in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:"""
)

#5) LLM
llm = Ollama(model="llama3.2", temperature=0)

def format_docs(docs_):
    return "\n\n".join(d.page_content for d in docs_)

#6) RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

#7) Ask
print(rag_chain.invoke("what is Protostomes?"))


Pages: 11
Random page preview:
 11. Animal Kingdom – Characteristics of Classification y Direct: The young one resembles the adult and it does not undergo metamorphosis. Examples: Humans, birds Types of Eggs Based on the quantity of yolk, they are of three types: y Microlecithal: The amount of yolk in the egg is less. Examples: Eggs of Herdmania, Sea Urchin y Mesolecithal: The amount of yolk in the egg is moderate. Examples: Eggs of frog, fish, toad y Macrolecithal: The amount of yolk in the egg is high. Examples: Eggs of birds, reptiles, bony fishes. Based on the distribution of yolk in the cytoplasm of egg, they are of thr
Document splits: 16
Protostomes are animals whose mouth comes first when they grow inside their mommy's tummy, before their bottom (anus) does. This means that in these animals, the mouth develops first and then the anus grows later.

Examples of Protostomes are:

- Planaria
- Earthworms
- Snakes
- Pigeons

Does that make sense?
